# 06 — Clustering dos Indicadores de endurecimento (análise nova)

Agrupamento hierárquico (Ward) e validação com k-means sobre 145 itens × 10 indicadores ordinais (0–3).
Objetivo: verificar se a estrutura não-supervisionada confirma ou desafia a taxonomia de regimes.

Descoberta preliminar: `monocromatizacao` correlaciona fracamente com todos os outros indicadores (ρ 0.08–0.36).

**Fonte:** `data/processed/corpus_dataset.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import chi2_contingency
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/processed/corpus_dataset.csv')

INDICATORS = ['desincorporacao', 'rigidez_postural', 'dessexualizacao',
              'uniformizacao_facial', 'heraldizacao', 'enquadramento_arquitetonico',
              'apagamento_narrativo', 'monocromatizacao', 'serialidade', 'inscricao_estatal']

INDICATOR_LABELS = {
    'desincorporacao': 'Desincorporação', 'rigidez_postural': 'Rigidez postural',
    'dessexualizacao': 'Dessexualização', 'uniformizacao_facial': 'Uniformização facial',
    'heraldizacao': 'Heraldicização', 'enquadramento_arquitetonico': 'Enquadramento arq.',
    'apagamento_narrativo': 'Apagamento narrativo', 'monocromatizacao': 'Monocromatização',
    'serialidade': 'Serialidade', 'inscricao_estatal': 'Inscrição estatal',
}

# Filter to complete indicator data
df_complete = df.dropna(subset=INDICATORS).copy()
X = df_complete[INDICATORS].values

print(f"Itens com indicadores completos: {len(df_complete)}")
print(f"Regimes: {df_complete['regime_iconocratico'].value_counts().to_dict()}")
print(f"In-scope: {df_complete['in_scope'].value_counts().to_dict()}")

## 6.1 Mapa de correlação entre indicadores (Spearman)

In [ ]:
corr = df_complete[INDICATORS].corr(method='spearman')
corr_labeled = corr.copy()
corr_labeled.columns = [INDICATOR_LABELS[c] for c in corr.columns]
corr_labeled.index = [INDICATOR_LABELS[c] for c in corr.index]

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr_labeled, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Spearman ρ'})
ax.set_title('Correlação entre Indicadores de endurecimento (Spearman)')
plt.tight_layout()
plt.savefig('../data/processed/fig_10_indicator_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Highlight weak correlations
print("Indicadores com correlação fraca (ρ < 0.30) a qualquer outro:")
for i, c1 in enumerate(INDICATORS):
    weak = [c2 for j, c2 in enumerate(INDICATORS) if i != j and abs(corr.loc[c1, c2]) < 0.30]
    if weak:
        print(f"  {INDICATOR_LABELS[c1]}: {len(weak)} correlações fracas — {', '.join(INDICATOR_LABELS[w] for w in weak)}")

## 6.2 Clustering hierárquico (Ward) + Silhouette

In [ ]:
# Hierarchical clustering
Z = linkage(X, method='ward', metric='euclidean')

# Silhouette analysis
sil_scores = {}
for k in range(2, 9):
    labels = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X)
    sil_scores[k] = silhouette_score(X, labels)

best_k = max(sil_scores, key=sil_scores.get)
print(f"Silhouette scores: {', '.join(f'k={k}: {v:.3f}' for k, v in sil_scores.items())}")
print(f"Melhor k: {best_k} (silhouette = {sil_scores[best_k]:.3f})")

# Dendrogram colored by regime
regime_colors = {'fundacional': '#4C72B0', 'normativo': '#DD8452',
                 'militar': '#C44E52', 'contra-alegoria': '#55A868'}
leaf_colors = [regime_colors.get(r, '#999999') for r in df_complete['regime_iconocratico'].fillna('unknown')]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Dendrogram
ax = axes[0]
dendrogram(Z, ax=ax, truncate_mode='lastp', p=30, leaf_rotation=90,
           leaf_font_size=8, show_contracted=True)
ax.axhline(y=Z[-(best_k-1), 2], color='red', linestyle='--', label=f'Corte k={best_k}')
ax.set_xlabel('Itens (contraído)')
ax.set_ylabel('Distância (Ward)')
ax.set_title('Dendrograma — Clustering Hierárquico (Ward)')
ax.legend()

# Silhouette plot
ax = axes[1]
ax.plot(list(sil_scores.keys()), list(sil_scores.values()), 'o-', color='#C44E52', linewidth=2)
ax.axvline(x=best_k, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Número de clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Análise de Silhouette')
ax.set_xticks(range(2, 9))

plt.tight_layout()
plt.savefig('../data/processed/fig_11_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

## 6.3 Corte ótimo + Cross-tabulação com regimes

In [ ]:
# Cut at best_k
df_complete['cluster_hier'] = AgglomerativeClustering(n_clusters=best_k, linkage='ward').fit_predict(X)

# Cross-tab
ct = pd.crosstab(df_complete['cluster_hier'], df_complete['regime_iconocratico'],
                 margins=True, margins_name='Total')
print("Cross-tab: Cluster × Regime")
print(ct.to_string())

# Chi-squared test
ct_no_margin = pd.crosstab(df_complete['cluster_hier'], df_complete['regime_iconocratico'])
chi2, p, dof, expected = chi2_contingency(ct_no_margin)
n = ct_no_margin.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct_no_margin.shape) - 1)))

print(f"\nChi² = {chi2:.2f}, p = {p:.2e}, dof = {dof}")
print(f"Cramér's V = {cramers_v:.3f}")
print(f"Interpretação: {'associação forte' if cramers_v > 0.5 else 'associação moderada' if cramers_v > 0.3 else 'associação fraca'} entre clusters e regimes")

## 6.4 Validação com K-Means

In [ ]:
# K-Means at same k range
ari_scores = {}
for k in range(2, 9):
    km_labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    hier_labels = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X)
    ari_scores[k] = adjusted_rand_score(hier_labels, km_labels)

print("Adjusted Rand Index (hierarchical vs k-means):")
for k, ari in ari_scores.items():
    bar = '█' * int(ari * 20)
    print(f"  k={k}: {ari:.3f} {bar}")

# K-means at best_k
df_complete['cluster_km'] = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit_predict(X)
ari_best = adjusted_rand_score(df_complete['cluster_hier'], df_complete['cluster_km'])
print(f"\nARI no melhor k={best_k}: {ari_best:.3f}")
print(f"Interpretação: {'alta concordância' if ari_best > 0.7 else 'concordância moderada' if ari_best > 0.4 else 'baixa concordância'}")

## 6.5 Perfis dos clusters

In [ ]:
# Mean indicators per cluster
cluster_means = df_complete.groupby('cluster_hier')[INDICATORS].mean()
cluster_means.columns = [INDICATOR_LABELS[c] for c in cluster_means.columns]

fig, ax = plt.subplots(figsize=(14, 6))
cluster_means.T.plot(kind='bar', ax=ax, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Score médio (0–3)')
ax.set_xlabel('Indicador')
ax.set_title(f'Perfil dos Clusters (k={best_k}) — Média por Indicador')
ax.legend(title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/fig_12_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

# Print cluster sizes and regime composition
print(f"\nTamanho dos clusters:")
for c in range(best_k):
    subset = df_complete[df_complete['cluster_hier'] == c]
    regime_dist = subset['regime_iconocratico'].value_counts().to_dict()
    print(f"  Cluster {c}: n={len(subset)} — {regime_dist}")

## 6.6 Monocromatização: com e sem

In [ ]:
# Cluster WITH monocromatizacao
labels_with = AgglomerativeClustering(n_clusters=best_k, linkage='ward').fit_predict(X)

# Cluster WITHOUT monocromatizacao
ind_no_mono = [i for i, c in enumerate(INDICATORS) if c != 'monocromatizacao']
X_no_mono = X[:, ind_no_mono]
labels_without = AgglomerativeClustering(n_clusters=best_k, linkage='ward').fit_predict(X_no_mono)

ari_mono = adjusted_rand_score(labels_with, labels_without)
print(f"ARI (com vs sem monocromatização): {ari_mono:.3f}")
print(f"Interpretação: {'monocromatização não altera a estrutura' if ari_mono > 0.8 else 'monocromatização afeta significativamente a estrutura' if ari_mono < 0.5 else 'efeito moderado'}")

# Cross-tab
ct_mono = pd.crosstab(labels_with, labels_without)
print(f"\nCross-tab (com × sem):")
print(ct_mono.to_string())

## 6.7 Detecção de outliers

In [ ]:
from sklearn.metrics import silhouette_samples

sil_samples = silhouette_samples(X, df_complete['cluster_hier'])
df_complete['silhouette'] = sil_samples

# Bottom 10 outliers
outliers = df_complete.nsmallest(10, 'silhouette')[['id', 'title', 'regime_iconocratico', 'silhouette']]
print("Itens com menor silhouette (pior ajuste ao cluster):")
for _, row in outliers.iterrows():
    print(f"  {row['id']:20s} sil={row['silhouette']:.3f}  regime={row['regime_iconocratico']:15s}  {row['title'][:60]}")

# Items with negative silhouette (misclassified)
neg = df_complete[df_complete['silhouette'] < 0]
print(f"\nItens com silhouette negativa: {len(neg)}")

## 6.8 Verificação: in_scope vs comparanda

In [ ]:
# Do in_scope=False items cluster separately?
ct_scope = pd.crosstab(df_complete['cluster_hier'], df_complete['in_scope'])
print("Cross-tab: Cluster × In-Scope")
print(ct_scope.to_string())

# Percentage of out-of-scope per cluster
print("\n% out-of-scope por cluster:")
for c in ct_scope.index:
    total = ct_scope.loc[c].sum()
    out = ct_scope.loc[c].get(False, 0)
    print(f"  Cluster {c}: {out}/{total} = {100*out/total:.1f}% out-of-scope")

print("\nSíntese: se comparanda se concentram em poucos clusters, são estruturalmente distintos.")
print("Se estão espalhados, scope_role é administrativo, não estrutural.")

## 6.9 Síntese

### Achados:
- [Preencher após execução: melhor k, concordância com regimes, papel da monocromatização]

### Implicações para a tese:
- Se clusters ≈ regimes: taxonomia validada por dados
- Se clusters ≠ regimes: descoberta de subtipos ou reagrupamento inesperado
- A monocromatização como indicador estruturalmente distinto merece discussão no capítulo de métodos